# 04 — Companies Dimension (`companies.csv`)

## Purpose
Create the shared company dimension from the data model: one row per project company. Every other table (`market_prices`, `news_tickers`, `events`, `fundamentals`) links to it through `ticker`.

## Output
`data/processed/companies.csv`

| Column | Type | Description |
|---|---|---|
| `ticker` | string, **PK** | Uppercase stock symbol |
| `company_name` | string | Full company name |
| `sector` | string | Sector (Yahoo Finance style): Technology, Consumer Cyclical, Healthcare, Financial Services, Consumer Defensive, Energy |
| `industry` | string | Industry within the sector |
| `currency` | string | Trading currency (USD for all ten companies) |
| `cik` | integer | SEC Central Index Key (used to pull fundamentals and 8-K filings) |

## Sources
- `ticker`, `company_name`, `sector`, `industry` and `currency` are the team's agreed company table (edit the table below if it changes).
- `cik` comes from the SEC ticker map `https://www.sec.gov/files/company_tickers.json`. If the download fails (no internet, SEC blocking), built-in CIK values are used instead; when both are available they are cross-checked.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

OUTPUT_PATH = Path("../../data/processed/companies.csv")
PROCESSED_DIR = OUTPUT_PATH.parent

SEC_URL = "https://www.sec.gov/files/company_tickers.json"
# The SEC asks automated clients to identify themselves. Put your own contact here.
SEC_USER_AGENT = "MarketPulse student project your-email@example.com"

ALLOWED_SECTORS = {"Technology", "Consumer Cyclical", "Healthcare", "Financial Services", "Consumer Defensive", "Energy"}

## 1. Company table

The team's agreed list of the ten project companies. Change it here if it is revised.

In [ ]:
companies = pd.DataFrame(
    [
        ("AAPL", "Apple Inc.",              "Technology",         "Consumer Electronics",    "USD"),
        ("AMZN", "Amazon.com, Inc.",        "Consumer Cyclical",  "Internet Retail",         "USD"),
        ("JNJ",  "Johnson & Johnson",       "Healthcare",         "Pharmaceuticals",         "USD"),
        ("JPM",  "JPMorgan Chase & Co.",    "Financial Services", "Diversified Banking",     "USD"),
        ("KO",   "The Coca-Cola Company",   "Consumer Defensive", "Non-Alcoholic Beverages", "USD"),
        ("LLY",  "Eli Lilly and Company",   "Healthcare",         "Pharmaceuticals",         "USD"),
        ("MSFT", "Microsoft Corporation",   "Technology",         "Software\u2014Infrastructure", "USD"),
        ("NVDA", "NVIDIA Corporation",      "Technology",         "Semiconductors",          "USD"),
        ("TSLA", "Tesla, Inc.",             "Consumer Cyclical",  "Auto Manufacturers",      "USD"),
        ("XOM",  "Exxon Mobil Corporation", "Energy",             "Oil & Gas Integrated",    "USD"),
    ],
    columns=["ticker", "company_name", "sector", "industry", "currency"],
)
companies["ticker"] = companies["ticker"].str.strip().str.upper()
display(companies)

## 2. CIK numbers from the SEC ticker map

The built-in values are a fallback and a cross-check. If the SEC file is reachable and disagrees with them, the notebook stops so the difference can be reviewed. The one documented exception is `XOM` (see `PERIOD_CIK_OVERRIDES`).

In [ ]:
BUILTIN_CIK = {
    "AAPL": 320193,  "MSFT": 789019,  "NVDA": 1045810, "JPM": 19617,   "JNJ": 200406,
    "LLY": 59478,    "XOM": 34088,    "KO": 21344,     "AMZN": 1018724, "TSLA": 1318605,
}

# Tickers whose CIK is deliberately kept at the value that filed during the project period
# (2021-2025), even though the SEC file now lists a different one. Keep this list short and explained.
PERIOD_CIK_OVERRIDES = {
    "XOM": "corporate restructuring in 2026; the 2021-2025 filings are under the original CIK",
}

try:
    response = requests.get(SEC_URL, headers={"User-Agent": SEC_USER_AGENT}, timeout=30)
    response.raise_for_status()
    sec = pd.DataFrame(response.json()).T
    sec["ticker"] = sec["ticker"].astype(str).str.upper()
    sec["cik_str"] = sec["cik_str"].astype(int)
    # Keep EVERY CIK the SEC lists for a ticker (a ticker can appear more than once).
    sec_all = sec.groupby("ticker")["cik_str"].apply(lambda s: sorted(set(s))).to_dict()
    print(f"Downloaded the SEC ticker map ({len(sec_all):,} tickers).")
except Exception as exc:
    sec_all = {}
    print(f"Could not download the SEC file ({exc}). Using the built-in CIK values.")


def choose_cik(ticker):
    builtin = BUILTIN_CIK[ticker]
    listed = sec_all.get(ticker, [])
    if not listed:
        return builtin, "built-in (SEC file not available for this ticker)"
    if builtin in listed:
        note = "confirmed by SEC file" if len(listed) == 1 else f"built-in; SEC lists several: {listed}"
        return builtin, note
    if ticker in PERIOD_CIK_OVERRIDES:
        return builtin, f"kept on purpose: SEC file now lists {listed} ({PERIOD_CIK_OVERRIDES[ticker]})"
    return pd.NA, f"MISMATCH: built-in {builtin}, SEC file lists {listed}"


picked = companies["ticker"].map(choose_cik)
companies["cik"] = [p[0] for p in picked]
companies["cik_check"] = [p[1] for p in picked]
display(companies[["ticker", "cik", "cik_check"]])

if companies["cik"].isna().any():
    raise ValueError("SEC CIK values differ from the built-in ones (see the MISMATCH rows above). "
                     "Check the company on sec.gov/cgi-bin/browse-edgar, then correct BUILTIN_CIK "
                     "or, if the change is intentional, add it to PERIOD_CIK_OVERRIDES.")

companies["cik"] = companies["cik"].astype("Int64")
companies = companies[["ticker", "company_name", "sector", "industry", "currency", "cik"]]

## 3. Validate

In [ ]:
assert len(companies) == 10, f"Expected 10 project companies, found {len(companies)}"
assert companies["ticker"].is_unique, "ticker must be unique"
assert companies["ticker"].eq(companies["ticker"].str.upper()).all(), "ticker must be uppercase"
assert companies["cik"].notna().all(), "every company needs a CIK"
assert companies["cik"].is_unique, "cik must be unique"
assert set(companies["sector"]) <= ALLOWED_SECTORS, set(companies["sector"]) - ALLOWED_SECTORS
assert companies["company_name"].str.strip().ne("").all()
assert companies["industry"].notna().all() and companies["currency"].eq("USD").all()

# Foreign-key check: every ticker used by the other tables must exist here.
known = set(companies["ticker"])
for filename in ["market_clean.csv", "news_tickers.csv", "news_clean.csv", "events_clean.csv"]:
    path = PROCESSED_DIR / filename
    if not path.exists():
        print(f"- {filename}: not found, skipped")
        continue
    used = set(pd.read_csv(path, usecols=["ticker"], dtype=str)["ticker"].dropna().str.strip().str.upper())
    unknown = used - known
    if unknown:
        raise ValueError(f"{filename} contains tickers that are not in companies: {sorted(unknown)}")
    print(f"- {filename}: OK ({len(used)} tickers, all in companies)")

## 4. Export

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
companies.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(companies)} companies to {OUTPUT_PATH.resolve()}")
display(companies)

## Completion checklist

- [ ] The ten rows match the team's canonical ticker list.
- [ ] Sector, industry and currency values match the team's agreed table.
- [ ] `cik` values come from the SEC file, or the fallback was noted.
- [ ] Other tables' tickers all exist in `companies`.
- [ ] `companies.csv` shared with Person 4 for the warehouse and Power BI.